In [18]:
from dotenv import load_dotenv

"""
Reads a file named .env in your project folder and loads each KEY=value line into the environment variables of the running program. 
Your .env file should contain:
ANTHROPIC_API_KEY=your-key-here
"""
load_dotenv()

# Imports the Anthropic class from the official SDK (pip install anthropic) ....
from anthropic import Anthropic

"""
Creates the client object you use to call the API. You pass no key because the SDK automatically reads the ANTHROPIC_API_KEY environment 
variable, which is why load_dotenv() had to run before this line. The load_dotenv() call placed above the anthropic import isn't 
required for this to work, since the key is only read when Anthropic() runs. Putting all imports at the top is more conventional.
"""
client = Anthropic()

# FIX: Change from "claude-sonnet-5-0" to "claude-sonnet-5" or "claude-sonnet-5-5"
model = "claude-sonnet-5-5" 

"""
Calls the Messages API, the main endpoint for talking to Claude. It sends your request over the internet and waits for the full response. 
The result is stored in message.
"""
message = client.messages.create(
    # Tells the API which model to use. The name on the left is the parameter, and the model on the right is your variable.
    model=model,
    # A required cap on how long the reply can be. A token is a chunk of text, roughly three-quarters of an English word. 
    # If Claude reaches 1000 tokens, it stops mid-answer. It is a ceiling and not a target, so a short answer uses far fewer tokens.
    max_tokens=1000,

    
    #The conversation so far, as a list of message dictionaries. Each has a role ("user" for you, "assistant" for Claude) and content 
    #(the text). Here there is one user message. For a multi-turn chat, you append earlier messages to this list, because the API has no 
    #memory between calls. You send the whole history each time. The closing ) ends the create( call.
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [19]:
# Multi Conversation Flow ....

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system_prompt = None):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        system=system_prompt
    )  

    #return message.content[0].text
    # Collect only the text blocks; skip thinking blocks
    return "".join(block.text for block in message.content if block.type == "text")

    #         Part	                                   What it does
    # for block in message.content	    Walks through every block in the reply, in order
    # if block.type == "text"	        Keeps only blocks whose type is "text" and skips the rest
    # block.text	                    Reads the string from each kept block
    # "".join(...)	                    Glues the kept strings into one string (the "" means nothing is inserted between them)


    # NOTE: Without the system_prompt, the output is:
    # Answer is:  Solve it by isolating x.
    # **Step 1: Subtract 3 from both sides.**
    # 5x + 3 − 3 = 2 − 3
    # 5x = −1
    # **Step 2: Divide both sides by 5.**
    # x = −1/5
    # **Answer:** x = −1/5 (or −0.2)
    # **Check:** 5(−1/5) + 3 = −1 + 3 = 2 ✓




In [20]:
messages = []
system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step.
    """

add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages, system_prompt)

print("Answer is: ", answer)

Answer is:  Good one to work on! The goal is to get **x** alone on one side of the equation.

Right now, x is being multiplied by 5, and then 3 is being added to it. To undo those operations, we work backward, undoing the addition first and the multiplication second.

**Step 1:** Look at the equation: 5x + 3 = 2

What could you do to **both sides** to get rid of the "+ 3" on the left? Try it, and tell me what you get.
